## Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)
- Buat struktur direktori di HDFS, misalnya /tidarmart/raw/transaksi, /tidarmart/raw/pelanggan, /tidarmart/raw/produk, /tidarmart/stream_input, /tidarmart/curated, /tidarmart/models.
- Unggah seluruh data ke HDFS. Tampilkan bukti ukuran, jumlah blok, dan faktor replikasi (hdfs dfs -du -h, hdfs fsck).
- Baca seluruh data dari HDFS (hdfs://localhost:9000/...) bukan dari path lokal pada tahap-tahap berikutnya.
- Jelaskan: mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM pseudo-distributed dibanding klaster nyata

In [1]:
# Membuat struktur direktori
!hdfs dfs -mkdir -p user/asfadani/tidarmart/raw/transaksi
!hdfs dfs -mkdir -p user/asfadani/tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p user/asfadani/tidarmart/raw/produk
!hdfs dfs -mkdir -p user/asfadani/tidarmart/stream_input
!hdfs dfs -mkdir -p user/asfadani/tidarmart/curated
!hdfs dfs -mkdir -p user/asfadani/tidarmart/models

# cek direktori
!hdfs dfs -ls user/asfadani/tidarmart

Found 4 items
drwxr-xr-x   - asfadani supergroup          0 2026-11-23 23:11 user/asfadani/tidarmart/curated
drwxr-xr-x   - asfadani supergroup          0 2026-11-23 23:11 user/asfadani/tidarmart/models
drwxr-xr-x   - asfadani supergroup          0 2026-11-23 23:11 user/asfadani/tidarmart/raw
drwxr-xr-x   - asfadani supergroup          0 2026-11-23 23:11 user/asfadani/tidarmart/stream_input


In [2]:
# Unggah file lokal ke hdfs
!hdfs dfs -put -f /home/asfadani/praktikum-bigdata/data_proyek/transaksi user/asfadani/tidarmart/raw
!hdfs dfs -put -f /home/asfadani/praktikum-bigdata/data_proyek/stream_input user/asfadani/tidarmart
!hdfs dfs -put -f /home/asfadani/praktikum-bigdata/data_proyek/pelanggan.json user/asfadani/tidarmart/raw/pelanggan
!hdfs dfs -put -f /home/asfadani/praktikum-bigdata/data_proyek/produk.json user/asfadani/tidarmart/raw/produk

# Menampilkan bukti
!hdfs dfs -du -h user/asfadani/tidarmart
!hdfs fsck /user/asfadani/user/asfadani/tidarmart -files -blocks -locations

0        0        user/asfadani/tidarmart/curated
0        0        user/asfadani/tidarmart/models
193.8 M  193.8 M  user/asfadani/tidarmart/raw
9.4 M    9.4 M    user/asfadani/tidarmart/stream_input
Connecting to namenode via http://localhost:9870/fsck?ugi=asfadani&files=1&blocks=1&locations=1&path=%2Fuser%2Fasfadani%2Fuser%2Fasfadani%2Ftidarmart
FSCK started by asfadani (auth:SIMPLE) from /127.0.0.1 for path /user/asfadani/user/asfadani/tidarmart at Mon Nov 23 23:34:27 WIB 2026

/user/asfadani/user/asfadani/tidarmart <dir>
/user/asfadani/user/asfadani/tidarmart/curated <dir>
/user/asfadani/user/asfadani/tidarmart/models <dir>
/user/asfadani/user/asfadani/tidarmart/raw <dir>
/user/asfadani/user/asfadani/tidarmart/raw/pelanggan <dir>
/user/asfadani/user/asfadani/tidarmart/raw/pelanggan/pelanggan.json 6308351 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1524836457-127.0.1.1-1788400594758:blk_1073741921_1097 len=6308351 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-f

>Data mentah dan data bersih harus dipisah karena untuk kerapian dan kemudahan dalam mengaksesnya. Data mentah beersifat immutable (tidak boleh dirubah) karena berfungsi sebagai sumber kebenaran utama. Dimana jika saat proses pengolahan terdapat kesalahan, masih ada titik 0 datanya yang bisa diolah kembali. Selain itu, jika file jadi dengan file mentah disatukan kita akan bingung jika suatu saat ingin membutuhkan filenya untuk sebuah projek. Selain itu juga menyimpan file dengan kondisi berbeda juga menjadi salah satu syarat clean architecture di dunia pemrograman. Pembagian penyimpanan data ini juga memudahkan dalam pengerjaan kolaborasi, sehingga orang lain yang akan mengakses datanya tidak kebingungan. Dalam pelacakan perubahan dari datanya juga menjadi mudah jika datanya disimpan ke folder yang berbeda. Selain itu juga membuat proses lebih efisien pada pipeline project, untuk mengatur data diambil dari mana dan akan disimpan di mana nantinya.

>Maksud dari replikasi 1 adalah jumlah replikasi (copy/redudan) data yang disimpan. Dalam indstri nyata bidang bigdata data wajib direplikasi ke beberapa tempat (umumnya 3 DataNode) untuk mengamankan data jika ada salah satu yang bermasalah. Data tidak akan langsung error karena sudah ada backupnya. Dalam VM psudeo distributed ini hanya meniru cara kerja di dunia nyata industri bigdata dengan menerapkan di project ini. Dan karena hanya jalan di 1 mesin VM, maka replikasi yang dipakai hanya 1 (menyesuaikan tempat).